# Verificación de configuración segura en AWS

Este notebook valida la configuración del Laboratorio 19 mediante
credenciales temporales del perfil local `tesis`.

No contiene claves de acceso, contraseñas, tokens, identificadores
de cuenta ni datos clínicos.

In [1]:
from pathlib import Path
import json
import boto3

PROFILE = "tesis"
REGION = "us-east-2"
BUCKET = "lab19-cdb-seguro-20261126-m4p7x2"
POLICY_PATH = Path("politica_minima.json")

session = boto3.Session(
    profile_name=PROFILE,
    region_name=REGION,
)

credentials = session.get_credentials()
sts = session.client("sts")
s3 = session.client("s3")

identity = sts.get_caller_identity()
iam_user = identity["Arn"].rsplit("/", 1)[-1]

assert credentials is not None
assert credentials.method == "login"
assert iam_user == "alejandro-lab19"

print(f"Perfil: {PROFILE}")
print(f"Región: {session.region_name}")
print(f"Método de credenciales: {credentials.method}")
print(f"Usuario IAM: {iam_user}")

Perfil: tesis
Región: us-east-2
Método de credenciales: login
Usuario IAM: alejandro-lab19


## 1. Bloqueo de acceso público

In [2]:
public_block = s3.get_public_access_block(
    Bucket=BUCKET
)["PublicAccessBlockConfiguration"]

required_settings = {
    "BlockPublicAcls",
    "IgnorePublicAcls",
    "BlockPublicPolicy",
    "RestrictPublicBuckets",
}

assert required_settings.issubset(public_block)
assert all(public_block[name] for name in required_settings)

for name in sorted(required_settings):
    print(f"{name}: {public_block[name]}")

print("Resultado: todo el acceso público está bloqueado.")

BlockPublicAcls: True
BlockPublicPolicy: True
IgnorePublicAcls: True
RestrictPublicBuckets: True
Resultado: todo el acceso público está bloqueado.


## 2. Cifrado predeterminado del bucket

In [3]:
encryption = s3.get_bucket_encryption(
    Bucket=BUCKET
)["ServerSideEncryptionConfiguration"]["Rules"]

algorithms = {
    rule["ApplyServerSideEncryptionByDefault"]["SSEAlgorithm"]
    for rule in encryption
}

assert algorithms == {"AES256"}

print(f"Algoritmos configurados: {sorted(algorithms)}")
print("Resultado: cifrado SSE-S3 habilitado.")

Algoritmos configurados: ['AES256']
Resultado: cifrado SSE-S3 habilitado.


## 3. Estructura y objetos almacenados

In [4]:
response = s3.list_objects_v2(Bucket=BUCKET)

objects = {
    item["Key"]: item["Size"]
    for item in response.get("Contents", [])
}

expected_files = {
    "publico/archivo_publico.txt",
    "privado/archivo_privado.txt",
}

assert expected_files.issubset(objects)

for key in sorted(objects):
    print(f"{key}: {objects[key]} bytes")

print("Resultado: están presentes los archivos público y privado.")

privado/: 0 bytes
privado/archivo_privado.txt: 104 bytes
publico/: 0 bytes
publico/archivo_publico.txt: 105 bytes
Resultado: están presentes los archivos público y privado.


## 4. Cifrado de los archivos de prueba

In [5]:
for key in sorted(expected_files):
    metadata = s3.head_object(
        Bucket=BUCKET,
        Key=key,
    )

    encryption_method = metadata.get(
        "ServerSideEncryption"
    )

    assert encryption_method == "AES256"

    print(
        f"{key}: "
        f"{metadata['ContentLength']} bytes, "
        f"cifrado={encryption_method}"
    )

print("Resultado: ambos archivos están cifrados con SSE-S3.")

privado/archivo_privado.txt: 104 bytes, cifrado=AES256


publico/archivo_publico.txt: 105 bytes, cifrado=AES256
Resultado: ambos archivos están cifrados con SSE-S3.


## 5. Validación de la política de privilegio mínimo

In [6]:
with POLICY_PATH.open(
    mode="r",
    encoding="utf-8",
) as policy_file:
    policy = json.load(policy_file)

assert policy["Version"] == "2012-10-17"
assert len(policy["Statement"]) == 1

statement = policy["Statement"][0]

assert statement["Effect"] == "Allow"
assert statement["Action"] == "s3:GetObject"
assert statement["Resource"] == (
    "arn:aws:s3:::"
    "lab19-cdb-seguro-20261126-m4p7x2/"
    "publico/*"
)
assert statement["Action"] != "s3:*"
assert statement["Resource"] != "*"

print(json.dumps(policy, indent=2, ensure_ascii=False))
print("Resultado: política mínima válida.")

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "LeerSoloObjetosPublicos",
      "Effect": "Allow",
      "Action": "s3:GetObject",
      "Resource": "arn:aws:s3:::lab19-cdb-seguro-20261126-m4p7x2/publico/*"
    }
  ]
}
Resultado: política mínima válida.


## Conclusión

La configuración comprobada utiliza autenticación temporal,
mantiene bloqueado el acceso público, cifra los objetos almacenados
y limita la política propuesta a la lectura de objetos bajo
`publico/`.